In [1]:
from influxdb_client import InfluxDBClient
import pandas as pd

# InfluxDB connection
url = "http://localhost:8086"
token = "nvv7pf6Kh7BWNI2gggEd7TrzA11j_WM_uW1lkBTtZytlcHx6JMZaumfDbSjvU5PN-Oi0CDG8-PUtBWYFKzEe0w=="
org = "B403Lab"
bucket = "TMS_Magnet_Proto_20260921"

# Query settings
start_time = "2026-09-21T21:35:00Z"
stop_time = "2026-09-22T00:57:00Z"
measurement = "TMS_Mag"

query = f'''
from(bucket: "{bucket}")
    |> range(start: {start_time}, stop: {stop_time})
    |> filter(fn: (r) => r["_measurement"] == "{measurement}")
    |> pivot(
        rowKey: ["_time"],
        columnKey: ["_field"],
        valueColumn: "_value"
    )
'''

with InfluxDBClient(url=url, token=token, org=org) as client:
    df = client.query_api().query_data_frame(query=query)

# A query can return a list when multiple tables are produced
if isinstance(df, list):
    df = pd.concat(df, ignore_index=True)

# Remove Flux metadata columns
df = df.drop(
    columns=["result", "table", "_start", "_stop", "_measurement"],
    errors="ignore"
)

# Sort chronologically
df = df.sort_values("_time")

output_file = measurement + "_data_"+ bucket[-8:]+ ".csv"
df.to_csv(output_file, index=False)

print(f"Downloaded {len(df)} rows")
print(f"Saved to: {output_file}")

Downloaded 41176 rows
Saved to: TMS_Mag_data_20260921.csv
